# Anàlisi d'Ocupació d'Aules (Nivell 1)
Aquest quadern respon a la primera pregunta guia del Nivell 1: el mapa de calor d'ocupació de cada aula per dia i franja, i per semestre, incloent-hi les sessions sense aula.

S'ha establert un ordre cronològic per dies de la setmana (L, M, X, J, V) i hores d'inici per poder llegir correctament l'evolució al llarg de la setmana.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar dades
sessions = pd.read_csv('sessions.csv')

# Neteja de dades: tractar sessions sense aula
sessions['aula_plot'] = sessions['aula_partida'].fillna('SENSE AULA')

# Funció per generar el mapa de calor per semestre
def plot_heatmap(semestre_codi, titol):
    df_semestre = sessions[sessions['semestre'] == semestre_codi].copy()
    
    # Agrupar per aula, dia i hora d'inici
    ocupacio = df_semestre.groupby(['aula_plot', 'dia', 'h_ini']).size().reset_index(name='sessions')
    
    # Establir l'ordre correcte dels dies de la setmana
    dies_ordre = ['L', 'M', 'X', 'J', 'V']
    ocupacio['dia'] = pd.Categorical(ocupacio['dia'], categories=dies_ordre, ordered=True)
    
    # Ordenar el dataframe per dia i després per hora d'inici
    ocupacio = ocupacio.sort_values(['dia', 'h_ini'])
    
    # Crear una clau combinada per l'eix X (Dia + Hora)
    ocupacio['franja'] = ocupacio['dia'].astype(str) + " " + ocupacio['h_ini']
    
    # Guardar l'ordre exacte de les franges per reindexar després del pivot
    ordre_franges = ocupacio['franja'].unique()
    
    # Pivotar per format heatmap
    pivot_df = ocupacio.pivot(index='aula_plot', columns='franja', values='sessions').fillna(0)
    
    # Reindexar per mantenir l'ordre cronològic (L -> V, matí -> vespre)
    pivot_df = pivot_df.reindex(columns=ordre_franges)
    
    # Dibuixar
    plt.figure(figsize=(20, 15))
    sns.heatmap(pivot_df, cmap='YlOrRd', cbar_kws={'label': 'Nombre de classes programades'})
    plt.title(f'Mapa de calor d\'ocupació: {titol}')
    plt.xlabel('Franja Horària (Ordenada per Dies i Hores)')
    plt.ylabel('Aula')
    plt.tight_layout()
    plt.show()

# Executar per al primer i segon semestre
print("Generant mapa de calor pel 1r Semestre...")
plot_heatmap('C/1', '1r Semestre')

print("Generant mapa de calor pel 2n Semestre...")
plot_heatmap('C/2', '2n Semestre')
